In [1]:
!pip install -q sentence-transformers faiss-cpu

In [2]:
!pip install -q peft bitsandbytes accelerate "transformers<5"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = '/content/drive/MyDrive/story_narrator'
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
GENRES = ['bedtime story', 'animal tale', 'adventure', 'magical story']

def make_prompt(genre, theme, characters, length):
    return (f'### Instruction:\nWrite a {length} {genre} for children.\n'
            f'Characters: {characters}\nTheme: {theme}\n\n### Story:\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={'': 0})
model = PeftModel.from_pretrained(base, f'{BASE}/models/story_lora')
model.eval()
print('LLM ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM ready


In [4]:
import re, json, requests, numpy as np, torch

BOOKS = {
    'Grimm': 'https://www.gutenberg.org/cache/epub/2591/pg2591.txt',
    'Andersen': 'https://www.gutenberg.org/cache/epub/1597/pg1597.txt',
}

def fetch_book(url):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    r.encoding = 'utf-8'
    text = r.text
    s = re.search(r'\*\*\* ?START OF[^\n]*\n', text)
    if s:
        text = text[s.end():]
    e = re.search(r'\*\*\* ?END OF', text)
    if e:
        text = text[:e.start()]
    return text

def chunk_text(text, source, min_chars=300, max_chars=800):
    paras = [re.sub(r'\s+', ' ', p).strip() for p in re.split(r'\n\s*\n', text)]
    chunks, cur = [], ''
    for p in paras:
        if not p:
            continue
        if len(cur) + len(p) + 1 <= max_chars:
            cur = (cur + ' ' + p).strip()
        else:
            if len(cur) >= min_chars:
                chunks.append({'source': source, 'text': cur})
            cur = p[:max_chars]
    if len(cur) >= min_chars:
        chunks.append({'source': source, 'text': cur})
    return chunks

chunks = []
for name, url in BOOKS.items():
    try:
        chunks += chunk_text(fetch_book(url), name)
    except Exception as e:
        print(f'Could not download {name}: {e}')
        print('Download the .txt from gutenberg.org on your computer and upload it with google.colab.files.upload().')

print('Total chunks:', len(chunks))
with open(f'{BASE}/data/fairy_chunks.json', 'w') as f:
    json.dump(chunks, f)
print(chunks[len(chunks) // 2]['text'][:400])

Total chunks: 988
When old Mr Fox was dead, the wolf came as a suitor, and knocked at the door, and the cat who was servant to Mrs Fox, opened it for him. The wolf greeted her, and said: ‘Good day, Mrs Cat of Kehrewit, How comes it that alone you sit? What are you making good?’ The cat replied: ‘In milk I’m breaking bread so sweet, Will you be my guest, and eat?’ ‘No, thank you, Mrs Cat,’ answered the wolf. ‘Is Mrs


In [5]:
from sentence_transformers import SentenceTransformer

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=dev)
emb = embedder.encode([c['text'] for c in chunks], batch_size=64,
                      show_progress_bar=True, normalize_embeddings=True).astype('float32')
np.save(f'{BASE}/data/fairy_embeddings.npy', emb)

try:
    import faiss
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    USE_FAISS = True
except ImportError:
    USE_FAISS = False
print('Embeddings:', emb.shape, '| FAISS:', USE_FAISS)

def retrieve(query, k=2):
    q = embedder.encode([query], normalize_embeddings=True).astype('float32')
    if USE_FAISS:
        scores, ids = index.search(q, k)
        ids, scores = ids[0], scores[0]
    else:
        sims = emb @ q[0]
        ids = np.argsort(-sims)[:k]
        scores = sims[ids]
    return [(chunks[int(i)], float(s)) for i, s in zip(ids, scores)]

for c, s in retrieve('animal story about friendship and kindness', k=2):
    print(round(s, 3), c['source'], '|', c['text'][:200], '\n')

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embeddings: (988, 384) | FAISS: True
0.483 Grimm | A shepherd’s dog had a master who took no care of him, but often let him suffer the greatest hunger. At last he could bear it no longer; so he took to his heels, and off he ran in a very sad and sorro 

0.476 Grimm | A farmer had a horse that had been an excellent faithful servant to him: but he was now grown too old to work; so the farmer would give him nothing more to eat, and said, ‘I want you no longer, so tak 



In [6]:
def short_snippet(text, max_chars=300):
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars]
    last = max(cut.rfind('.'), cut.rfind('!'), cut.rfind('?'))
    return cut[:last + 1] if last > 100 else cut

def make_prompt_rag(genre, theme, characters, length, inspiration):
    return (f'### Instruction:\nWrite a {length} {genre} for children.\n'
            f'Characters: {characters}\nTheme: {theme}\n'
            f'Inspiration (borrow the mood and style, do not copy the words): {inspiration}\n\n'
            f'### Story:\n')

def _generate(prompt, temperature, max_new_tokens):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, top_p=0.9, repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    if text and text[-1] not in '.!?"':
        cut = max(text.rfind('.'), text.rfind('!'), text.rfind('?'))
        if cut > 0:
            text = text[:cut + 1]
    return text

def generate_story_v2(genre, theme, characters, length='short', use_rag=False, temperature=0.8):
    """Returns (story, inspiration_used)."""
    max_new = 250 if length == 'short' else 400
    inspiration = ''
    if use_rag:
        hits = retrieve(f'{genre} about {theme} with {characters}', k=1)
        inspiration = short_snippet(hits[0][0]['text'])
        prompt = make_prompt_rag(genre, theme, characters, length, inspiration)
    else:
        prompt = make_prompt(genre, theme, characters, length)
    return _generate(prompt, temperature, max_new), inspiration

In [7]:

args_ = ('magical story', 'bravery', 'a small dragon named Ember, a baker named Tom', 'short')

plain, _ = generate_story_v2(*args_, use_rag=False)
rag, used = generate_story_v2(*args_, use_rag=True)

print('=== WITHOUT RAG ===\n', plain, '\n')
print('=== INSPIRATION USED ===\n', used, '\n')
print('=== WITH RAG ===\n', rag)

=== WITHOUT RAG ===
 Once upon a time, there was a small dragon named Ember. Ember lived in a big forest with many other animals. One day, Ember wanted to bake something delicious. She asked her friend the baker named Tom if he could help her.

Tom said yes and they went to Ember's cave to start baking. They mixed flour, eggs, sugar, and milk together. Then they put the mixture into a pot and heated it up on the fire. After a few minutes, the pot started to bubble and steam. Ember felt proud of herself because she made a delicious cake!

But when they were done baking, Tom told Ember that she needed to be careful not to burn the cake while it is cooking. Ember nodded her head and promised to stay close by so she wouldn't get burned. From then on, Ember always remembered to be careful and brave like a true dragon! 

=== INSPIRATION USED ===
 THE STORY OF A MOTHER A mother sat there with her little child. She was so downcast, so afraid that it should die! 

=== WITH RAG ===
 One day, she

In [8]:
import os, tempfile, inspect
import numpy as np, soundfile as sf
import gradio as gr

THEMES = ['kindness', 'friendship', 'bravery', 'curiosity', 'honesty']

def write_story(genre, theme, characters, length, creativity, use_rag, progress=gr.Progress()):
    if not characters or not characters.strip():
        raise gr.Error('Please describe at least one character.')
    progress(0.2, desc='Writing your story...')
    story, used = generate_story_v2(genre, theme or 'kindness', characters.strip(), length,
                                    use_rag=use_rag, temperature=creativity)
    if not story:
        raise gr.Error('The story came out empty. Please try again.')
    return story, (used or 'Story library was not used.')

def narrate_story(story_text, voice1, voice2, consent, progress=gr.Progress()):
    if not consent:
        raise gr.Error('Please confirm that you have permission from this person to use their voice.')
    if not story_text or len(story_text.strip()) < 20:
        raise gr.Error('Please write or paste a story first.')
    voices = [v for v in (voice1, voice2) if v]
    if not voices:
        raise gr.Error('Please upload or record a voice clip.')

    tmp_dir = tempfile.mkdtemp()
    clean, notes = [], []
    try:
        for i, v in enumerate(voices):
            p = os.path.join(tmp_dir, f'voice_{i}.wav')
            try:
                notes += preprocess_voice(v, p)
            except ValueError as e:
                raise gr.Error(str(e))
            clean.append(p)
        progress(0.1, desc='Voice prepared')

        segments = split_segments(story_text)
        sr = tts.synthesizer.output_sample_rate
        pause = np.zeros(int(sr * 0.4), dtype=np.float32)
        chunks_audio = []
        for i, seg in enumerate(segments):
            progress(0.1 + 0.85 * i / len(segments), desc=f'Narrating part {i + 1} of {len(segments)}')
            wav = tts.tts(text=seg, speaker_wav=clean if len(clean) > 1 else clean[0], language='en')
            chunks_audio.append(np.array(wav, dtype=np.float32))
            chunks_audio.append(pause)
        out_path = os.path.join(tmp_dir, 'story.wav')
        sf.write(out_path, np.concatenate(chunks_audio), sr)
    finally:
        for p in clean:
            if os.path.exists(p):
                os.remove(p)       # never keep the uploaded voice

    status = '✅ Your narrated story is ready. Press play, or use the download button on the player.'
    if notes:
        status += '\n\n⚠️ ' + ' '.join(sorted(set(notes)))
    return out_path, status

In [9]:
CSS = """
.hero {text-align:center; padding:20px 12px; border-radius:16px;
       background:linear-gradient(135deg,#6d5dfc,#c86dd7); color:white;}
.hero h1 {margin:0; font-size:2rem; color:white;}
.hero p {margin:8px 0 0 0; opacity:0.95; color:white;}
.footer-note {text-align:center; opacity:0.75; font-size:0.9rem;}
"""
THEME = gr.themes.Soft(primary_hue='violet', secondary_hue='pink')

# Works with both older and newer Gradio versions
blocks_kw = {}
if 'theme' in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kw = dict(theme=THEME, css=CSS)

with gr.Blocks(title='AI Story Narrator', **blocks_kw) as demo:
    gr.HTML('<div class="hero"><h1>📖 AI Story Narrator</h1>'
            '<p>Create a story, then hear it narrated in the voice of someone you love.</p></div>')

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown('### 1. Choose your story')
            genre = gr.Dropdown(GENRES, value='bedtime story', label='Genre')
            theme = gr.Dropdown(THEMES, value='kindness', label='Theme (you can type your own)',
                                allow_custom_value=True)
            characters = gr.Textbox(value='a little girl named Mira, a sleepy owl', label='Characters',
                                    placeholder='e.g. a brave boy named Arjun, a talking parrot')
            length = gr.Radio(['short', 'medium'], value='short', label='Length')
            creativity = gr.Slider(0.5, 1.1, value=0.8, step=0.05, label='Creativity',
                                   info='Lower is safer and simpler, higher is more imaginative.')
            use_rag = gr.Checkbox(value=False, label='Use the fairy tale library for inspiration (RAG)')
            btn_story = gr.Button('✍️ Write my story', variant='primary')

        with gr.Column(scale=6):
            gr.Markdown('### 2. Read and edit your story')
            story_box = gr.Textbox(lines=14, interactive=True, label='Story (you can edit it before narration)')
            with gr.Accordion('Library passage used for inspiration', open=False):
                inspiration_box = gr.Textbox(lines=4, interactive=False, label='')

    gr.Markdown('### 3. Choose the narrator voice')
    with gr.Row():
        voice1 = gr.Audio(sources=['upload', 'microphone'], type='filepath',
                          label='Voice clip (10 to 30 seconds)')
        voice2 = gr.Audio(sources=['upload', 'microphone'], type='filepath',
                          label='Optional second clip (helps keep the accent)')
    with gr.Accordion('Tips for a good voice clip', open=False):
        gr.Markdown(
            '- One person only, in a quiet room, with no music or echo.\n'
            '- Speak naturally and clearly, like telling a story. Keep the phone a hand-width from the mouth.\n'
            '- 15 to 30 seconds works best. Two different clips can improve accent and style.\n'
            '- Only use the voice of someone who has given you permission.'
        )
    consent = gr.Checkbox(label='I have permission from this person to use their voice')
    btn_narrate = gr.Button('🎙️ Narrate my story', variant='primary')
    status = gr.Markdown()
    audio_out = gr.Audio(label='Narrated story (AI-generated voice)', type='filepath')

    gr.Markdown('The narration is AI-generated. Uploaded voice clips are deleted right after use and are not stored.',
                elem_classes=['footer-note'])

    btn_story.click(write_story, [genre, theme, characters, length, creativity, use_rag],
                    [story_box, inspiration_box])
    btn_narrate.click(narrate_story, [story_box, voice1, voice2, consent], [audio_out, status])

launch_kw = {}
if 'theme' in inspect.signature(demo.launch).parameters and not blocks_kw:
    launch_kw = dict(theme=THEME, css=CSS)

demo.queue()
demo.launch(share=True, **launch_kw)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://<temporary-gradio-link>

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Notes**
- The public link works while this notebook is running. Open it on your laptop or phone.
- If the app shows an error, read the message in the red box. Common ones: no consent tick, clip too short, or no voice uploaded.
- If the old app is still running, stop its cell before launching this one.
- Gradio versions differ slightly. If a cell complains about an unknown argument (for example `sources` or `allow_custom_value`), send me the error and I will adjust it.